# Section 1: System Verification
Checking GPU availability and VRAM status.

In [6]:
!nvidia-smi

Thu Apr 16 06:38:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Section 2: GPU Memory Management
Identify and terminate orphan Python processes to free up VRAM.

In [7]:
import os, signal
current_pid = os.getpid()
for line in os.popen('ps -ef | grep python').readlines():
    fields = line.split()
    pid = int(fields[1])
    # Kill non-jupyter python processes to clear old sessions
    if pid != current_pid and 'jupyter' not in line:
        print(f"Cleaning orphan process {pid}...")
        try: os.kill(pid, signal.SIGKILL)
        except: pass

!nvidia-smi

Cleaning orphan process 88...
Cleaning orphan process 1126...
Cleaning orphan process 1128...
Thu Apr 16 06:38:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P0             25W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |   

# Section 3: Consolidated Installations
**IMPORTANT**: Run this and then **Restart Session** (Runtime > Restart session) if you encounter any import errors.

In [8]:
%pip install -U transformers accelerate bitsandbytes datasets tqdm pandas matplotlib langchain-community langchain-text-splitters faiss-gpu-cu12 pyngrok uvicorn nest_asyncio

# Section 4: Global Imports & Configuration

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from google.colab import drive
from langchain_core.documents import Document as LangchainDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import nest_asyncio
import uvicorn
from fastapi import FastAPI, Request
from pyngrok import ngrok
import asyncio

# Configuration
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN"
DATASET_PATH = "/content/drive/MyDrive/Registration_info_dataset123.txt"
EMBEDDING_MODEL_NAME = "thenlper/gte-small"

# Section 5: Persistent Storage
Mounting Google Drive to access the dataset.

In [10]:
drive.mount('/content/drive')

Mounted at /content/drive


# Section 6: Data Engineering
Loading the registration dataset from Google Drive.

In [11]:
try:
    with open("/content/drive/MyDrive/Registration_info_dataset123.txt", "r") as fp:
        s = fp.read().split("\n\n\n\n")
    
    RAW_KNOWLEDGE_BASE = [
        LangchainDocument(page_content=doc)
        for doc in tqdm(s, desc="Loading Documents")
    ]
    print(f"Successfully loaded {len(RAW_KNOWLEDGE_BASE)} documents.")
except FileNotFoundError:
    print(f"ERROR: {DATASET_PATH} not found. Check your Drive path.")


Loading Documents:   0%|          | 0/1633 [00:00<?, ?it/s]

Successfully loaded 1633 documents.


# Section 7: Text Preprocessing
Splitting the documents into manageable chunks for the LLM.

In [12]:
MARKDOWN_SEPARATORS = ["\n#{1,6}", "```\n", "\n\\*\\*\\*+\n", "\n---+\n", "\n__+\n", "\n\n", "\n", " ", ""]
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    add_start_index=True,
    strip_whitespace=True,
    separators=MARKDOWN_SEPARATORS,
)
docs_processed = []
for doc in RAW_KNOWLEDGE_BASE:
    docs_processed += text_splitter.split_documents([doc])
print(f"Created {len(docs_processed)} chunks for indexing.")

Created 4260 chunks for indexing.


# Section 8: Vector Engine
Initializing Embeddings and building the FAISS database.

In [13]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    multi_process=True,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

KNOWLEDGE_VECTOR_DATABASE = FAISS.from_documents(
    docs_processed,
    embedding_model,
    distance_strategy=DistanceStrategy.COSINE,
)

/tmp/ipython-input-4250615280.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# Section 9: Model Mastery
Loading Llama-3.1-8B with 4-bit quantization on T4 GPU.

In [14]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

# Section 10: RAG Reasoning Pipeline
Expert persona and retrieval functions.

In [15]:
prompt_chat = [
    {
        "role": "system",
        "content": """You are an expert Registration Information Assistant. 
Using the provided Context, give a helpful and legal-minded response.

Guidelines:
1. **Direct Answer**: Focus on solving the user's registration query.
2. **Reference**: Cite the specific Act, Department, or Document found in the Context.
3. **Constraint**: If not in context, say: 'I don\'t have official information on this specific topic.'
4. **Disclaimer**: Add a note that this is for guidance purposes only."""
    },
    {
        "role": "user",
        "content": "Context:\n{context}\n---\nQuestion: {question}"
    }
]

RAG_PROMPT_TEMPLATE = tokenizer.apply_chat_template(prompt_chat, tokenize=False, add_generation_prompt=True)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def answer_query(question, k=3):
    # Similarity Search
    context_docs = KNOWLEDGE_VECTOR_DATABASE.similarity_search(question, k=k)
    context_text = "\n".join([doc.page_content for doc in context_docs])
    
    # Format and Generate
    full_prompt = RAG_PROMPT_TEMPLATE.format(context=context_text, question=question)
    
    # FIX: We added return_full_text=False to automatically hide Llama's system tags
    response = pipe(full_prompt, max_new_tokens=512, do_sample=False, temperature=0.0, return_full_text=False)
    
    # Now you can just cleanly return the generated text!
    return response[0]['generated_text'].strip()


# Section 11: Chatbot API Deployment
Hosting the chatbot via Ngrok and FastAPI.

In [ ]:
app = FastAPI()

@app.post("/query")
async def chat(request: Request):
    data = await request.json()
    query = data.get("query")
    response = answer_query(query)
    return {"response": response}

# Deployment Logic
nest_asyncio.apply()
!pkill -f ngrok
ngrok.set_auth_token("2wE9yU0hK3W0H4iBM3B2OxNWPkA_7yxsy911qCZvTeBF2T4Nk")
public_url = ngrok.connect(8000)
print(f"\n>>> Your Chatbot API is LIVE at: {public_url.public_url}")
print("Ensure your server.jsx points to /query endpoint.")

async def start_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    await server.serve()

if __name__ == "__main__":
    # Add server task to the notebook loop
    loop = asyncio.get_event_loop()
    loop.create_task(start_server())

                                                                                                    
>>> Your Chatbot API is LIVE at: https://a624-136-109-102-214.ngrok-free.app
Ensure your server.jsx points to /query endpoint.


INFO:     Started server process [860]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     183.82.51.10:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     183.82.51.10:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     183.82.51.10:0 - "POST / HTTP/1.1" 404 Not Found
INFO:     183.82.51.10:0 - "POST / HTTP/1.1" 404 Not Found
